The purpose of this notebook is only to show that things work after refactoring.

And that the new results are consistent with the previous results.

In [1]:
from pathlib import Path

import gwfast.network as network
import gwfast.waveforms as waveforms
from gwfast.gwfastGlobals import detectors as det_dict, detPath
from gwfast.detector import Detector
from gwfast.AGN_lensed_signal import AGNLensedGWSignal
from gwfast.new_signal import NewGWSignal
import gwfast.fisherTools as fTools

/users/hin-wai.leong/src/AGN-gwfast/gwfast/waveforms.py:33: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


TEOBResumS is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH


In [10]:
events = {
    'Mc':np.array([30, 30]), 'eta':np.array([0.24, 0.24]), 'dL':np.array([2, 2]), 
    'theta':np.array([2.34, 2.34]), 'phi':np.array([5.43, 5.43]), 
    'iota':np.array([0.99*np.pi/2, 0.99*np.pi/2]), 'psi':np.array([1, 1]), 
    'tGPS':np.array([0, 0]), 'phase':np.array([2.8, 2.8]), 
    'chi1z':np.array([1e-3, 1e-3]), 'chi2z':np.array([1e-3, 1e-3]), 
    'R_orbit':np.array([200, 50]), 'M_lz':np.array([10e4, 10e5]), 'src_pos':np.array([0.7, 0.1])
}
events = {key: val.astype(np.float64) for key, val in events.items()}

H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
L1 = Detector('L1', **det_dict['L1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
V1 = Detector('V1', **det_dict['Virgo'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/avirgo_O5low_NEW.txt')

PhenomD = waveforms.IMRPhenomD()
H1_AGN = AGNLensedGWSignal(wf_model=PhenomD, detector=H1, fmin=10)
L1_AGN = AGNLensedGWSignal(wf_model=PhenomD, detector=L1, fmin=10)
V1_AGN = AGNLensedGWSignal(wf_model=PhenomD, detector=V1, fmin=10)
HL_signals = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN})
HLV_signals = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN, 'V1': V1_AGN})

f_array = np.geomspace(20, 400, num=500)
f_array = np.broadcast_to(f_array, (events['Mc'].shape[0], 500))

event_1_strains = L1_AGN.GWstrain(f_array.T, events)
print(event_1_strains.T.shape)

fisher_mats = HLV_signals.FisherMatr(events, res=1000)

Initializing jax...
Jax local device count: 8
Jax device count: 8
odict_keys(['Mc', 'phase', 'chi1z', 'chi2z', 'chis', 'chia', 'dL', 'eta', 'iota', 'phi', 'psi', 'Lambda1', 'Lambda2', 'tcoal', 'theta', 'chi1x', 'chi2x', 'chi1y', 'chi2y', 'ecc', 'R_orbit', 'M_lz', 'src_pos'])
Initializing jax...
Jax local device count: 8
Jax device count: 8
odict_keys(['Mc', 'phase', 'chi1z', 'chi2z', 'chis', 'chia', 'dL', 'eta', 'iota', 'phi', 'psi', 'Lambda1', 'Lambda2', 'tcoal', 'theta', 'chi1x', 'chi2x', 'chi1y', 'chi2y', 'ecc', 'R_orbit', 'M_lz', 'src_pos'])
Initializing jax...
Jax local device count: 8
Jax device count: 8
odict_keys(['Mc', 'phase', 'chi1z', 'chi2z', 'chis', 'chia', 'dL', 'eta', 'iota', 'phi', 'psi', 'Lambda1', 'Lambda2', 'tcoal', 'theta', 'chi1x', 'chi2x', 'chi1y', 'chi2y', 'ecc', 'R_orbit', 'M_lz', 'src_pos'])
Adding tcoal from tGPS
(2, 500)
Computing Fisher for H1...
odict_keys(['Mc', 'eta', 'dL', 'theta', 'phi', 'iota', 'psi', 'tGPS', 'phase', 'chi1z', 'chi2z', 'R_orbit', 'M_lz

In [3]:
fisher_mats = HL_signals.FisherMatr(events, res=1000)

Computing Fisher for H1...
odict_keys(['Mc', 'eta', 'dL', 'theta', 'phi', 'iota', 'psi', 'tGPS', 'phase', 'chi1z', 'chi2z', 'R_orbit', 'M_lz', 'src_pos', 'tcoal'])
Computing Fisher for L1...
odict_keys(['Mc', 'eta', 'dL', 'theta', 'phi', 'iota', 'psi', 'tGPS', 'phase', 'chi1z', 'chi2z', 'R_orbit', 'M_lz', 'src_pos', 'tcoal'])
Done.


In [4]:
fisher_mats.shape, len(events.keys()), events.keys()

((15, 15, 2),
 15,
 dict_keys(['Mc', 'eta', 'dL', 'theta', 'phi', 'iota', 'psi', 'tGPS', 'phase', 'chi1z', 'chi2z', 'R_orbit', 'M_lz', 'src_pos', 'tcoal']))

In [5]:
# Well somehow this `tGPS` thing is added to the parameters
# but it is not used in the waveform...
# Need to figure out what's going on
keys = list(events.keys())
tGPS_idx = keys.index('tGPS')
keys.remove('tGPS')
reduced_fisher_mats = np.delete(fisher_mats, (tGPS_idx), axis=0)
reduced_fisher_mats = np.delete(reduced_fisher_mats, (tGPS_idx), axis=1)

print("\nThe Fisher matrices:")
fisher_mats_iter = np.moveaxis(reduced_fisher_mats, 2, 0)
for matrix in fisher_mats_iter:
    row = f'{"":8}   ' + '  '.join([f'{col_key:^11}' for col_key in keys])
    print(row)
    for rdx, row_key in enumerate(keys):
        row = f'{row_key:>8}  '
        for cdx, col_key in enumerate(keys):
            row += f'{matrix[rdx][cdx]:+11.3e}  '
        print(row)
    print('--------------------')
print('=================================================================')

reduced_cov_mats, ie = fTools.CovMatr(reduced_fisher_mats)
covar_mats_iter = np.moveaxis(reduced_cov_mats, 2, 0)

print("\nThe Covariance matrices:")
for matrix in covar_mats_iter:
    row = f'{"":8}   ' + '  '.join([f'{col_key:^11}' for col_key in keys])
    print(row)
    for rdx, row_key in enumerate(keys):
        row = f'{row_key:>8}  '
        for cdx, col_key in enumerate(keys):
            row += f'{matrix[rdx][cdx]:+11.3e}  '
        print(row)


The Fisher matrices:
               Mc           eta          dL          theta         phi         iota          psi        phase       chi1z        chi2z       R_orbit       M_lz        src_pos       tcoal   
      Mc   +1.614e+03   -3.250e+04   +1.216e+00   -1.491e+03   +1.272e+03   +3.979e+02   +3.329e-01   +1.850e+02   -6.143e+03   -2.875e+03   -7.381e-01   -9.769e-01   -1.449e+05   -1.093e+05  
     eta   -3.250e+04   +6.640e+05   -4.859e+01   +3.180e+04   -2.744e+04   -8.152e+03   -7.174e+01   -3.693e+03   +1.255e+05   +5.779e+04   +1.528e+01   +2.053e+01   +3.045e+06   +2.347e+06  
      dL   +1.216e+00   -4.859e+01   +4.955e+00   +4.285e+01   +2.630e+01   +6.361e+00   +1.380e+01   +1.641e-01   -7.265e+00   -3.598e+00   -4.157e-03   +3.710e-04   +6.020e+01   -1.020e+02  
   theta   -1.491e+03   +3.180e+04   +4.285e+01   +2.465e+03   -1.303e+03   -2.085e+02   +2.411e+02   -1.656e+02   +6.038e+03   +2.579e+03   +7.041e-01   +1.181e+00   +1.753e+05   +1.384e+05  
     phi   +1.27

In [6]:
for key, val in zip(keys, np.sqrt(np.diag(covar_mats_iter[0]))):
    print(key, '--', val)

Mc -- 1.6510976001327748714
eta -- 0.14578741369908790264
dL -- 116.651396199234982085
theta -- 86.04144583600090241
phi -- 88.07141130487902514
iota -- 4.3719591480185082183
psi -- 75.53556828504189133
phase -- 10.994362876876326871
chi1z -- 10.415977667491293324
chi2z -- 16.0469601204777354
R_orbit -- 1104.2804924372948401
M_lz -- 118634.56493926657332
src_pos -- 0.7995216600445318465
tcoal -- 2.2007289789438570118


In [7]:
# Test No-lensing signal
H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
L1 = Detector('L1', **det_dict['L1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
V1 = Detector('V1', **det_dict['Virgo'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/avirgo_O5low_NEW.txt')

PhenomD = waveforms.IMRPhenomD()
H1_BBH = NewGWSignal(wf_model=PhenomD, detector=H1, fmin=10)
L1_BBH = NewGWSignal(wf_model=PhenomD, detector=L1, fmin=10)
V1_BBH = NewGWSignal(wf_model=PhenomD, detector=V1, fmin=10)
HL_signals_BBH = network.DetNet({'H1': H1_BBH, 'L1': L1_BBH})
HLV_signals_BBH = network.DetNet({'H1': H1_BBH, 'L1': L1_BBH, 'V1': V1_BBH})

bbh_events = events.copy()
for key in ('R_orbit', 'M_lz', 'src_pos'):
    bbh_events.pop(key)

Initializing jax...
Jax local device count: 8
Jax device count: 8
odict_keys(['Mc', 'phase', 'chi1z', 'chi2z', 'chis', 'chia', 'dL', 'eta', 'iota', 'phi', 'psi', 'Lambda1', 'Lambda2', 'tcoal', 'theta', 'chi1x', 'chi2x', 'chi1y', 'chi2y', 'ecc', 'R_orbit', 'M_lz', 'src_pos'])
Initializing jax...
Jax local device count: 8
Jax device count: 8
odict_keys(['Mc', 'phase', 'chi1z', 'chi2z', 'chis', 'chia', 'dL', 'eta', 'iota', 'phi', 'psi', 'Lambda1', 'Lambda2', 'tcoal', 'theta', 'chi1x', 'chi2x', 'chi1y', 'chi2y', 'ecc', 'R_orbit', 'M_lz', 'src_pos'])
Initializing jax...
Jax local device count: 8
Jax device count: 8
odict_keys(['Mc', 'phase', 'chi1z', 'chi2z', 'chis', 'chia', 'dL', 'eta', 'iota', 'phi', 'psi', 'Lambda1', 'Lambda2', 'tcoal', 'theta', 'chi1x', 'chi2x', 'chi1y', 'chi2y', 'ecc', 'R_orbit', 'M_lz', 'src_pos'])


In [8]:
fisher_mats_BBH = HLV_signals_BBH.FisherMatr(bbh_events)

# We will figure out what is going on later:
keys = list(bbh_events.keys())
tGPS_idx = keys.index('tGPS')
keys.remove('tGPS')
reduced_fisher_mats = np.delete(fisher_mats_BBH, (tGPS_idx), axis=0)
reduced_fisher_mats = np.delete(reduced_fisher_mats, (tGPS_idx), axis=1)

# keys = list(bbh_events.keys())
print("\nThe Fisher matrices:")
fisher_mats_iter = np.moveaxis(reduced_fisher_mats, 2, 0)
for matrix in fisher_mats_iter:
    row = f'{"":8}   ' + '  '.join([f'{col_key:^11}' for col_key in keys])
    print(row)
    for rdx, row_key in enumerate(keys):
        row = f'{row_key:>8}  '
        for cdx, col_key in enumerate(keys):
            row += f'{matrix[rdx][cdx]:+11.3e}  '
        print(row)
    print('--------------------')

print('=================================================================')
cov_mats, ie = fTools.CovMatr(reduced_fisher_mats)
covar_mats_iter = np.moveaxis(cov_mats, 2, 0)

print("\nThe Covariance matrices:")
for matrix in covar_mats_iter:
    row = f'{"":8}   ' + '  '.join([f'{col_key:^11}' for col_key in keys])
    print(row)
    for rdx, row_key in enumerate(keys):
        row = f'{row_key:>8}  '
        for cdx, col_key in enumerate(keys):
            row += f'{matrix[rdx][cdx]:+11.3e}  '
        print(row)

Computing Fisher for H1...
odict_keys(['Mc', 'eta', 'dL', 'theta', 'phi', 'iota', 'psi', 'tGPS', 'phase', 'chi1z', 'chi2z', 'tcoal'])
Computing Fisher for L1...
odict_keys(['Mc', 'eta', 'dL', 'theta', 'phi', 'iota', 'psi', 'tGPS', 'phase', 'chi1z', 'chi2z', 'tcoal'])
Computing Fisher for V1...
odict_keys(['Mc', 'eta', 'dL', 'theta', 'phi', 'iota', 'psi', 'tGPS', 'phase', 'chi1z', 'chi2z', 'tcoal'])
Done.

The Fisher matrices:
               Mc           eta          dL          theta         phi         iota          psi        phase       chi1z        chi2z        tcoal   
      Mc   +6.163e+03   -1.262e+05   -9.022e-01   -9.804e+02   +1.953e+03   -6.721e+01   +5.715e+01   +6.620e+02   -2.385e+04   -1.103e+04   -4.410e+05  
     eta   -1.262e+05   +2.611e+06   -7.256e+01   +2.098e+04   -4.212e+04   +1.282e+03   -1.176e+03   -1.350e+04   +4.933e+05   +2.250e+05   +9.514e+06  
      dL   -9.022e-01   -7.256e+01   +1.783e+01   +2.648e+01   +1.643e+01   +1.988e+00   +3.838e+00   -5.483e-1

In [9]:
for key, val in zip(keys, np.sqrt(np.diag(covar_mats_iter[0]))):
    print(key, '--', val)

Mc -- 1.4757039733737796877
eta -- 0.08974261696514239154
dL -- 0.3433805275503621039
theta -- 0.08182729829567299776
phi -- 0.12261473061931495082
iota -- 0.105642655602181431614
psi -- 0.26427195175754647112
phase -- 9.32030170862138632
chi1z -- 6.7756758919689022277
chi2z -- 10.337344270743296271
tcoal -- 0.08101350713046684632
